# Assessment Notebook

In [1]:
# Numerical arrays
import numpy as np

## Problem 1: Representing SHA-256 Data

SHA-256 uses different types of data when processing a message and producing a hash. These include the original message, 32-bit words, 512-bit message blocks and the final 256-bit hash value.

Python provides different ways to store and work with these values. For example, `bytes` can be used for the original message, while NumPy provides `np.uint32` for the 32-bit words used in the SHA-256 calculations.

In this problem, I will demonstrate how each type of data can be represented in Python. I will also explain why I have chosen each representation and show small examples of how they work.

The main reference for this problem is the NIST Secure Hash Standard (FIPS 180-4). I will use Sections 3.1, 5.1.1, 5.2.1 and 6.2.2 to understand the data sizes, message format and SHA-256 calculations.

Reference:
https://doi.org/10.6028/NIST.FIPS.180-4

#### 32-bit words

A 32-bit word contains exactly 32 bits and can represent an unsigned integer between 0 and 4,294,967,295. SHA-256 uses 32-bit words throughout its calculations, so it is important that these values are represented correctly.

Python integers can store numbers larger than 32 bits, so they do not automatically have the same size as the words used by SHA-256. NumPy provides `np.uint32`, which represents an unsigned 32-bit integer. I have used this in my example to make sure the word uses the correct data type.

I also looked at a Stack Overflow discussion about fixed-width integers in Python. It explains why using a fixed-size data type can be useful when performing calculations that depend on the number of bits.

References:

- NIST FIPS 180-4, Section 3.1 – Bit Strings and Integers:
  https://doi.org/10.6028/NIST.FIPS.180-4

- NumPy – Data types:
  https://numpy.org/doc/stable/user/basics.types.html

- Stack Overflow – Fixed-width integer types in Python:
  https://stackoverflow.com/questions/31145143/fixed-width-integer-types-e-g-uint32-in-python

In [2]:
# Represent a 32-bit word
word = np.uint32(0x12345678)

print("Word:", hex(int(word)))
print("Data type:", word.dtype)

assert word.dtype == np.uint32
assert int(word) == 0x12345678

print("Test passed!")

Word: 0x12345678
Data type: uint32
Test passed!


#### Sequences of 32-bit words

SHA-256 uses sequences of 32-bit words when processing messages and performing its calculations. For example, each 512-bit message block is divided into sixteen 32-bit words.

In Python, these sequences can be represented using a list or a NumPy array. I have chosen a NumPy array because it allows all the values to use the same data type. By setting the data type to `np.uint32`, each value is stored as an unsigned 32-bit integer.

I looked at the NumPy array creation documentation to understand how to create an array with a specific data type. This is useful because it allows me to create a sequence of words that can be accessed using their index.

References:

- NIST FIPS 180-4, Section 6.2.2 – SHA-256 Hash Computation:
  https://doi.org/10.6028/NIST.FIPS.180-4

- NumPy – Array creation:
  https://numpy.org/doc/stable/user/basics.creation.html

- Stack Overflow – Fixed-width integer types in Python:
  https://stackoverflow.com/questions/31145143/fixed-width-integer-types-e-g-uint32-in-python

In [3]:
# Represent a sequence of 32-bit words
words = np.array(
    [0x12345678, 0xABCDEF01, 0x87654321],
    dtype=np.uint32
)

print([hex(int(word)) for word in words])

assert len(words) == 3
assert words.dtype == np.uint32

print("Test passed!")

['0x12345678', '0xabcdef01', '0x87654321']
Test passed!


#### Input messages

SHA-256 can process messages of different lengths. In Python, an input message can be represented using a `bytes` object because SHA-256 processes the actual bytes of the message rather than Python characters.

A normal Python string (`str`) must first be converted into bytes before it can be used as input. The `encode()` method can be used for this, and UTF-8 is a suitable choice because it supports normal ASCII text as well as non-ASCII characters.

For example, the string "abc" can be converted into a bytes object using UTF-8 encoding. The resulting bytes can then be used by the SHA-256 algorithm.

I also looked at a Stack Overflow discussion about encoding strings before using them with Python hashing functions. It helped explain why a normal string needs to be converted into bytes first.

References:

- NIST FIPS 180-4, Section 5.1.1 – SHA-1, SHA-224 and SHA-256 Padding:
  https://doi.org/10.6028/NIST.FIPS.180-4

- Python – str.encode():
  https://docs.python.org/3/library/stdtypes.html#str.encode

- Stack Overflow – Encoding strings before hashing:
  https://stackoverflow.com/questions/7585307/how-to-correct-typeerror-unicode-objects-must-be-encoded-before-hashing

In [4]:
# Represent an input message as bytes
message = "abc".encode("utf-8")

print(message)
print("Length:", len(message), "bytes")

assert message == b"abc"
assert isinstance(message, bytes)

print("Test passed!")

b'abc'
Length: 3 bytes
Test passed!


#### 512-bit message blocks

SHA-256 processes messages in blocks of 512 bits. Each block contains 64 bytes, which can also be divided into sixteen 32-bit words.

In Python, a block can be represented using a `bytes` object. NumPy can then be used to interpret these bytes as sixteen 32-bit unsigned integers.

I have used `np.frombuffer()` for this because it allows the bytes to be read directly as a NumPy array. I have also used `>u4` as the data type. This means that each value is an unsigned four-byte integer using big-endian byte order.

I looked at the NumPy documentation and a Stack Overflow discussion about converting bytes into 32-bit integers. Both explain how bytes can be interpreted as numerical values using `frombuffer()`. This is useful for understanding how I will convert message blocks into words in Problem 4.

The example below uses 64 bytes to demonstrate the structure of one complete block. The actual message padding and splitting into blocks will be implemented later.

References:

- NIST FIPS 180-4, Section 5.2.1 – SHA-1, SHA-224 and SHA-256 Message Parsing:
  https://doi.org/10.6028/NIST.FIPS.180-4

- NumPy – numpy.frombuffer():
  https://numpy.org/doc/stable/reference/generated/numpy.frombuffer.html

- Stack Overflow – Converting bytes into 32-bit integer values:
  https://stackoverflow.com/questions/77887703/how-to-convert-a-big-bytes-object-to-a-list-of-32-bit-integer-values

In [5]:
# Represent a 512-bit block as sixteen 32-bit words
block = bytes(range(64))

block_words = np.frombuffer(
    block, dtype=">u4"
).astype(np.uint32)

print("Block size:", len(block) * 8, "bits")
print("Number of words:", len(block_words))

assert len(block_words) == 16
assert int(block_words[0]) == 0x00010203

print("Test passed!")

Block size: 512 bits
Number of words: 16
Test passed!


#### The final 256-bit hash value

SHA-256 always produces a hash value that is 256 bits long, regardless of the length of the original message. This means that the final hash contains 32 bytes.

In Python, the hash can be represented using a `bytes` object. It can also be converted into a hexadecimal string using the `hex()` method. A hexadecimal representation contains 64 characters because each character represents four bits.

I looked at Python's `hashlib` documentation to understand how SHA-256 produces its final digest. It explains the difference between `digest()`, which returns the hash as bytes, and `hexdigest()`, which returns it as a hexadecimal string.

In the example below, I use `hashlib` to generate an actual SHA-256 hash of the message "abc". I will use this again in Problem 6 when testing my own SHA-256 implementation.

References:

- NIST FIPS 180-4, Section 6.2.2 – SHA-256 Hash Computation:
  https://doi.org/10.6028/NIST.FIPS.180-4

- Python – hashlib:
  https://docs.python.org/3/library/hashlib.html

- Python – bytes.hex():
  https://docs.python.org/3/library/stdtypes.html#bytes.hex

In [6]:
# Generate and represent a SHA-256 hash
import hashlib

hash_value = hashlib.sha256(b"abc").digest()

print("Hash:", hash_value.hex())
print("Hash size:", len(hash_value) * 8, "bits")

assert len(hash_value) == 32
assert len(hash_value.hex()) == 64

print("Test passed!")

Hash: ba7816bf8f01cfea414140de5dae2223b00361a396177a9cb410ff61f20015ad
Hash size: 256 bits
Test passed!


#### Big-endian integers

Big-endian is a way of storing numbers where the most significant byte comes first. For example, the 32-bit integer `0x12345678` is stored as four bytes: `12 34 56 78`.

SHA-256 uses big-endian byte ordering when interpreting message blocks as 32-bit words. This means that the order of the bytes is important because interpreting them in the wrong order would produce a different integer.

Python provides `to_bytes()` to convert an integer into bytes and `int.from_bytes()` to convert bytes back into an integer. Both methods allow the byte order to be specified.

I also looked at a Stack Overflow discussion about endianness. It explains why the same bytes can produce different integer values depending on whether they are interpreted as big-endian or little-endian. This helped me understand why I need to specify the byte order when working with SHA-256.

References:

- NIST FIPS 180-4, Section 3.1 – Bit Strings and Integers:
  https://doi.org/10.6028/NIST.FIPS.180-4

- Python – int.to_bytes():
  https://docs.python.org/3/library/stdtypes.html#int.to_bytes

- Stack Overflow – Why do we need endianness here?:
  https://stackoverflow.com/questions/33692321/why-do-we-need-endianness-here

In [7]:
# Convert an integer into big-endian bytes
word = 0x12345678

big_endian = word.to_bytes(4, byteorder="big")
converted_word = int.from_bytes(big_endian, byteorder="big")

print("Bytes:", big_endian)
print("Integer:", hex(converted_word))

assert converted_word == word

print("Test passed!")

Bytes: b'\x124Vx'
Integer: 0x12345678
Test passed!


## Problem 2: SHA-256 Bitwise Operations

Problem 2 involves implementing the Boolean and bitwise functions used by SHA-256:

- `Ch(x, y, z)`

- `Maj(x, y, z)`

- `Sigma0(x)` - written as  Σ_0^256(x) in the standard

- `Sigma1(x)` — written as Σ_1^256(x) in the standard

- `sigma0(x)` — written as σ_0^256(x) in the standard

- `sigma1(x)` — written as σ_1^256(x) in the standard

Use NumPy so that the values involved in these operations are represented as 32-bit unsigned integers.

For each function:

- provide an appropriate docstring,
- explain the operation in Markdown,
- implement the function directly from that definition,
- provide tests on appropriate inputs.

### Ch(x, y, z)
#### Explanation:

`&` performs bitwise AND.

`^` performs bitwise XOR.

`~` performs bitwise NOT.

The function chooses bits from y where x has a 1 and from z where x has a 0.

In [8]:
# Choice function Ch(x, y, z)
def Ch(x, y, z):
    """Return the SHA-256 choice function using 32-bit unsigned integers."""
    x = np.uint32(x)
    y = np.uint32(y)
    z = np.uint32(z)

    return (x & y) ^ (~x & z)

# Test function
x = np.uint32(0b1100)
y = np.uint32(0b1010)
z = np.uint32(0b0110)

result = Ch(x, y, z)

print(result)
print(bin(int(result)))

# Why: Ch selects bits from y when corresponding bit in x is 1, and bits from z when corresponding bit in x is 0. NumPy uint32 makes sure that the values use 32-bit unsigned ints.

10
0b1010


### Maj(x, y, z)
#### Explanation:

`&` performs bitwise AND.

`^` performs bitwise XOR.

The majority function takes three input bits and returns 1 if at least two of the corresponding bits are 1. Otherwise, it returns 0.

In [9]:
# Majority function Maj(x, y, z)
def Maj(x, y, z):
    """Return the SHA-256 majority function using 32-bit unsigned integers."""
    x = np.uint32(x)
    y = np.uint32(y)
    z = np.uint32(z)

    return (x & y) ^ (x & z) ^ (y & z)

# Test function
x = np.uint32(0b1100)
y = np.uint32(0b1010)
z = np.uint32(0b0110)

result = Maj(x, y, z)

print(result)
print(bin(int(result)))

# Why: Maj returns 1 for each bit position where at least two of the corresponding bits in x, y, and z are 1. NumPy uint32 makes sure that the values use 32-bit unsigned ints.

14
0b1110


### Sigma0(x)

#### Explanation:
`>>` performs a bitwise right shift.

`<<` performs a bitwise left shift.

`|` performs bitwise OR.

`^` performs bitwise XOR.

The Sigma function rotates the 32-bit input x right by 2, 13, and 22 bits then combines the three results using bitwise XOR.

In [10]:
# Sigma0 function Σ_0^256(x)
def Sigma0(x):
    """Return the SHA-256 Sigma0 function using 32-bit unsigned integers."""
    x = np.uint32(x)

    return (
        ((x >> np.uint32(2)) | (x << np.uint32(30))) ^
        ((x >> np.uint32(13)) | (x << np.uint32(19))) ^
        ((x >> np.uint32(22)) | (x << np.uint32(10)))
    )

# Test function
x = np.uint32(0b1100)

result = Sigma0(x)

print(result)
print(bin(int(result)))

# Why: Sigma0 rotates x right by 2, 13, and 22 bits then XORs the three rotated values together. NumPy uint32 makes sure that the values use 32-bit unsigned ints.

6303747
0b11000000011000000000011


### Sigma1(x)

#### Explanation:
`>>` performs a bitwise right shift.

`<<` performs a bitwise left shift.

`|` performs bitwise OR.

`^` performs bitwise XOR.

The Sigma1 function rotates the 32-bit input x right by 6, 11, and 25 bits then combines the three results using bitwise XOR.

In [11]:
# Sigma1 function Σ_1^256(x)
def Sigma1(x):
    """Return the SHA-256 Sigma1 function using 32-bit unsigned integers."""
    x = np.uint32(x)

    return (
        ((x >> np.uint32(6)) | (x << np.uint32(26))) ^
        ((x >> np.uint32(11)) | (x << np.uint32(21))) ^
        ((x >> np.uint32(25)) | (x << np.uint32(7)))
    )

# Test function
x = np.uint32(0b1100)

result = Sigma1(x)

print(result)
print(bin(int(result)))

# Why: Sigma1 rotates x right by 6, 11, and 25 bits then XORs the three rotated values together. NumPy uint32 makes sure that the values use 32-bit unsigned ints.

830473728
0b110001100000000000011000000000


### sigma0(x)

#### Explanation:
`>>` performs a bitwise right shift.

`<<` performs a bitwise left shift.

`|` performs bitwise OR.

`^` performs bitwise XOR.

The sigma0 function rotates the 32-bit input x right by 7 and 18 bits then shifts x right by 3 bits then combines the three results using bitwise XOR.

In [12]:
# sigma0 function σ_0^256(x)
def sigma0(x):
    """Return the SHA-256 sigma0 function using 32-bit unsigned integers."""
    x = np.uint32(x)

    return (
        ((x >> np.uint32(7)) | (x << np.uint32(25))) ^
        ((x >> np.uint32(18)) | (x << np.uint32(14))) ^
        (x >> np.uint32(3))
    )

# Test function
x = np.uint32(0b1100)

result = sigma0(x)

print(result)
print(bin(int(result)))

# Why: sigma0 rotates x right by 7 and 18 bits then shifts x right by 3 bits and XORs the three results. NumPy uint32 makes sure that the values use 32-bit unsigned ints.

402849793
0b11000000000110000000000000001


### sigma1(x)

#### Explanation:
`>>` performs a bitwise right shift.

`<<` performs a bitwise left shift.

`|` performs bitwise OR.

`^` performs bitwise XOR.

The sigma1 function rotates the 32-bit input x right by 17 and 19 bits, then shifts x right by 10 bits, and combines the three results using bitwise XOR.

In [13]:
# sigma1 function σ_1^256(x)
def sigma1(x):
    """Return the SHA-256 sigma1 function using 32-bit unsigned integers."""
    x = np.uint32(x)

    return (
        ((x >> np.uint32(17)) | (x << np.uint32(15))) ^
        ((x >> np.uint32(19)) | (x << np.uint32(13))) ^
        (x >> np.uint32(10))
    )

# Test function
x = np.uint32(0b1100)

result = sigma1(x)

print(result)
print(bin(int(result)))

# Why: sigma1 rotates x right by 17 and 19 bits then shifts x right by 10 bits and XORs the three results. NumPy uint32 makes sure that the values use 32-bit unsigned ints.

491520
0b1111000000000000000


## Problem 3: Generating the SHA-256 Constants

Problem 3 involves defining the initial hash values and round constants used by SHA-256 using fractional parts of square and cube roots of prime numbers.

Use Python and NumPy to generate the 64 round constants specified in the standard.

Your solution should:
1. write a function `primes(n)` that generates the first `n` prime numbers,
2. calculate the cube roots of the first 64 primes,
3. extract the fractional part of each cube root,
4. convert the resulting to a 32-bit unsigned integer,
5. display the resulting constants in hexadecimal,
6. compare your results with the constants in the Secure Hash Standard.

In [14]:
# Function to generate the first n prime numbers
def primes(n):
    prime_numbers = []
    num = 2

    while len(prime_numbers) < n:
        is_prime = True

        for i in range(2, int(np.sqrt(num)) + 1):
            if num % i == 0:
                is_prime = False
                break

        if is_prime:
            prime_numbers.append(num)

        num += 1

    return np.array(prime_numbers)

# Generate first 64 primes
p = primes(64)

print(p)

[  2   3   5   7  11  13  17  19  23  29  31  37  41  43  47  53  59  61
  67  71  73  79  83  89  97 101 103 107 109 113 127 131 137 139 149 151
 157 163 167 173 179 181 191 193 197 199 211 223 227 229 233 239 241 251
 257 263 269 271 277 281 283 293 307 311]


## Problem 4: Padding and Parsing Messages

Problem 4 involves write a generator function called `block_parse(msg)` that accepts a `bytes` object and yields the 512-bit message blocks required by SHA-256.

Each yielded block should contain sixteen 32-bit unsigned integers.

Implement the padding specified in sections 5.1.1 and 5.2.1 of the Secure Hash Standard, including:

- appending the `1` bit,

- appending the required zero bits,

- appending the original message length as a 64-bit integer,

- handling messages that require one or two final blocks,

- interpreting the message using the required byte order.

Test your implementation, including any boundary cases you can identify.

## Problem 5: The SHA-256 Compression Function

Problem 5 involves implementing the SHA-256 compression algorithm described in section 6.2.2 of the Secure Hash Standard.

Your implementation should explicitly construct the 64-word message schedule and perform the 64 rounds of the compression function.

Structure your implementation so that the major stages of the algorithm are identifiable in your code.

Explain:

- how the message schedule is constructed;
- the purpose of the eight working variables;
- how `Ch` and `Maj` are used;
- how the two Σ functions are used;
- how the round constants are incorporated;
- how the working variables are updated;
- how the compressed state is combined with the previous hash state.

Use your `block_parse` generator and the functions implemented in the previous problems.

## Problem 6: Complete SHA-256

Problem 6 involves combining your previous work into a function:

> sha256(msg)

The function should accept a `bytes` object and return its SHA-256 digest as a 64-character lowercase hexadecimal string.

Test your implementation against Python's `hashlib.sha256` using a range of inputs, including:

- the empty message,
- short ASCII strings,
- messages containing non-ASCII characters,
- messages whose lengths are close to padding boundaries,
- a longer message.